<a href="https://colab.research.google.com/github/Tanmay-Godse/Financial-Analytics-Data-Pipeline-with-Machine-Learning/blob/main/Run_LLM_on_T4_or_higher_GPU_via_VLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run opensource LLM's with VLLM server on COLAB GPU

## Things to keep in mind before starting:



*   While VLLM Server is running, do not goto **STEP-11** and rerun it, it'll crash the server and will become too technical to restart, instead kill the session, restart the kernal and run **STEP-4.1** first, then follow all the steps in sequence!

*   Not every model from HuggingFace would work here, the base T4_GPU **(Available in free version of colab)** only have ~15GB of VRAM, so plan accordingly.

*   If you have **Colab PRO**, you can leverage higher memory GPU like L4 (approx. 24GB VRAM), A100 (Var_1: approx. 40GB VRAM, Var_2: approx. 80GB VRAM), H100(approx. 80GB VRAM, **Strongest GPU** on colab currently), G4 (RTX A6000 Pro, approx. 96GB VRAM).

##**STEP-1**  Check GPU Stats:

In [39]:
!nvidia-smi

Sat Apr 25 18:11:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             36W /   70W |   12489MiB /  15360MiB |     79%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## **STEP-1.1** Solves broken path problems

In [3]:
%cd /content

/content


##**STEP-2** Install UV (Python Package Installer + Manager)

In [4]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 80.4 MB/s eta 0:00:00


## **STEP-3** Install Important Libs:

In [5]:
!uv pip install torch torchvision torchaudio openai transformers huggingface_hub accelerate safetensors --index-url https://download.pytorch.org/whl/cu121

Using Python 3.12.13 environment at: /usr
Checked 8 packages in 150ms


##**STEP-4** Install VLLM (LLM SERVER):

In [6]:
!uv pip install vllm

Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 1.76s
Prepared 52 packages in 29.87s
Uninstalled 5 packages in 499ms
Installed 52 packages in 485ms
 + anthropic==0.97.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==5.9.0
 + compressed-tensors==0.15.0.1
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.17.0
 + fastar==0.11.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + gguf==0.18.0
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.11.0
 + model-hosting-container-standards==0.1.14
 + msgspec==0.21.1
 + ninja==1.13.0
 - numba==0.60.0
 + numba==0.61.2
 + nvidia-cudnn-frontend==1.18.0
 + nvidia-cutlass-dsl==4.4.2
 + nvidia-cutlass-dsl-libs-base==4.4.2
 + openai-harmony==0.0.8
 + opentelemetry-ex

##**(STEP-4.1)** Delete folder only if path is not found in **STEP-8**!

In [7]:
!rm -rf /root/.cache/

##**STEP-5** Download Model from HuggingFace:

In [9]:
!hf download Qwen/Qwen3-4B-Instruct-2507

Fetching 13 files: 100% 13/13 [01:21<00:00,  6.24s/it]
Download complete: : 8.06GB [01:21, 147MB/s]              ✓ Downloaded
  path: /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554
Download complete: : 8.06GB [01:21, 99.4MB/s]


##**STEP-6** Check folder size (OPTIONAL)

In [10]:
!du -sh /root/.cache/

7.6G	/root/.cache/


##**STEP-7** Check pressent working directory :

In [11]:
!pwd

/content


## **STEP-8** This is the model download path:

In [12]:
%cd ../root/.cache/huggingface/hub

/root/.cache/huggingface/hub


##**(OPTIONAL STEP-1)** Check Current working directory, updated!

In [13]:
!pwd

/root/.cache/huggingface/hub


##**(OPTIONAL STEP-2)** Check all available/downloaded Models from huggingface (to download model, run : 'hf download "model card name" ').

In [14]:
!ls ~/.cache/huggingface/hub/

CACHEDIR.TAG  models--Qwen--Qwen3-4B-Instruct-2507


##**STEP-9** Get Model Snapshot ID (Unique Hash ID provided by HuggingFace)

In [15]:
!ls ~/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/

cdbee75f17c01a7cc42f958dc650907174af0554


##**STEP-10** Set the model path to that unique snapshot (Caution: This snapshot hash ID might get changed, please update model path manually from above cell output [ID], This value will be different for every LLM model!)

In [16]:
model_path = "/root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554"

##**STEP-11** Run VLLM Server in backend and use it via API endpoint

In [17]:
!python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen3-4B-Instruct-2507 \
    --dtype float16 \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.80 \
    --trust-remote-code \
    --enforce-eager \
    > vllm.log 2>&1 &

## **STEP-11.1** Post check on server stats

In [25]:
!ps aux | grep vllm

root        3371 27.3  9.3 7977644 1238176 ?     Sl   18:01   0:24 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --dtype float16 --max-model-len 8192 --gpu-memory-utilization 0.80 --trust-remote-code --enforce-eager
root        3861  0.0  0.0   7376  3576 ?        S    18:02   0:00 /bin/bash -c ps aux | grep vllm
root        3863  0.0  0.0   6484  2572 ?        S    18:02   0:00 grep vllm


##**(OPTIONAL STEP-3)** Post checks on running VLLM server.


*   **You can re-run this cell frequently to check the VLLM server updates**
* **You can also re-run and check STEP-1 for frequent GPU updates**


* Once you see **Application startup complete** by re-running this cell, you are good to hit API endpoint created by VLLM server!

In [38]:
!tail -n 50 vllm.log

(APIServer pid=3371) INFO 04-25 18:06:32 [api_server.py:592] Supported tasks: ['generate']
(APIServer pid=3371) WARNING 04-25 18:06:33 [model.py:1435] Default vLLM sampling parameters have been overridden by the model's `generation_config.json`: `{'temperature': 0.7, 'top_k': 20, 'top_p': 0.8}`. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
(APIServer pid=3371) INFO 04-25 18:06:37 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
(APIServer pid=3371) INFO 04-25 18:06:38 [api_server.py:596] Starting vLLM server on http://0.0.0.0:8000
(APIServer pid=3371) INFO 04-25 18:06:38 [launcher.py:37] Available routes are:
(APIServer pid=3371) INFO 04-25 18:06:38 [launcher.py:46] Route: /openapi.json, Methods: HEAD, GET
(APIServer pid=3371) INFO 04-25 18:06:38 [launcher.py:46] Route: /docs, Methods: HEAD, GET
(APIServer pid=3371) INFO 04-25 18:06:38 [launcher.py:46] Route: /

## **STEP-12** Ping to test API endpoint created by VLLM Server!
###Wait 3~5 mins after running **STEP-11**
### Till then You can run and check **(OPTIONAL STEP-3) and **STEP-1**
### Also you can check GPU RAM consumption in Resources if GUI info required!


In [36]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen3-4B-Instruct-2507","object":"model","created":1777140511,"owned_by":"vllm","root":"Qwen/Qwen3-4B-Instruct-2507","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-a4a29116038ff576","object":"model_permission","created":1777140511,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

## **STEP-13** Command to use the LLM via openAI endpoint created by running VLLM in backend, you can change your desired prompt by editing here : "Explain Schrödinger equation in depth"

###**SUGESTION**
###Until **STEP-12** output says **Connection Refused**, do not hit VLLM API endpoint!

###if crashed, goto **[STEP-4.1]** and redo steps.

In [37]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

resp = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=[{"role": "user", "content": "Explain Schrödinger equation in depth"}],
)

print(resp.choices[0].message.content)

The **Schrödinger equation** is the cornerstone of quantum mechanics, providing a mathematical description of how the quantum state of a physical system evolves over time. It replaces Newtonian mechanics in the microscopic world—where particles like electrons behave in ways that are fundamentally probabilistic and wave-like.

Let’s explore the Schrödinger equation in **depth**, covering its historical context, mathematical form, physical interpretation, types, solutions, and implications.

---

## 🔹 1. Historical Context

- In the early 20th century, physicists observed that electrons in atoms did not behave like classical particles orbiting a nucleus (as in Bohr’s model).
- Louis de Broglie proposed in 1924 that matter has wave-like properties—electrons, for instance, could be described as waves.
- Erwin Schrödinger, in 1926, developed a wave equation that described how these matter waves evolve in time. This equation became the foundation of wave mechanics.
- Schrödinger's equation i

#**That's all for now Folks, see you later!**


